# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [1]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


commit : d27c1c0
GPU    : Tesla T4
VRAM   : 14.6 GB


In [2]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[  ok  ] unit tests                                       119 passed in 5.03s

7 passed · 0 warnings · 0 failures

Ready to submit.


In [ ]:
# @title 3. Core pipeline — NB1 → NB5 (full evaluation)
import os, subprocess, sys
from pathlib import Path

COMPUTE_TIER = "T4"  # @param ["CPU","LAPTOP","T4","BIGGPU"]
STAGES = "nb1 nb2 nb3 nb4 nb5"  # @param {type:"string"}
# To reuse saved adapters, choose STAGES = "nb2 nb5".
# Explicit zero also overrides an EVAL_LIMIT=8 left in .env.
os.environ["COMPUTE_TIER"] = COMPUTE_TIER
os.environ["EVAL_LIMIT"] = "0"

from labkit import device
print(device.banner())

# Each stage gets its own process so GPU memory is released between stages.
# Save complete predictions while they are still available in notebook globals.
runner = r"""
import json, pathlib, runpy, sys
root = pathlib.Path.cwd()
sys.path.insert(0, str(root / "src"))
from labkit import evaluate as ev
stage = sys.argv[1]
paths = {
    "nb1": "01_data_and_mask.py", "nb2": "02_baselines.py",
    "nb3": "03_train_correct.py", "nb4": "04_misconfig_autopsy.py",
    "nb5": "05_evaluate_and_verdict.py", "nb6": "06_merge_and_serve.py",
}
state = runpy.run_path(str(root / "notebooks" / paths[stage]), run_name="__main__")
results = root / "results"
if stage == "nb2":
    rows = [dict(i=i, ticket=r["input"], label=r["label"],
                 baseline_a_pred=a, baseline_b_pred=b)
            for i, (r, a, b) in enumerate(zip(state["target"], state["preds_a"], state["preds_b"]))]
    payload = {"model": state["TIER"].model_id,
               "optimized_prompt_sha": state["frozen"]["optimized_prompt_sha"],
               "rows": rows}
    (results / "baseline_predictions.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
elif stage == "nb5":
    saved = json.loads((results / "baseline_predictions.json").read_text(encoding="utf-8"))
    assert saved["model"] == state["TIER"].model_id, "Baseline model mismatch"
    assert saved["optimized_prompt_sha"] == state["frozen"]["optimized_prompt_sha"], "Prompt mismatch"
    assert len(saved["rows"]) == len(state["target"]), "Eval length mismatch"
    rows = []
    for i, (r, p, b) in enumerate(zip(state["target"], state["preds_ft"], saved["rows"])):
        assert b["ticket"] == r["input"] and b["label"] == r["label"], "Eval content mismatch"
        ft = ev.triage_field_accuracy(p, r["label"])
        base = ev.triage_field_accuracy(b["baseline_b_pred"], r["label"])
        rows.append(dict(i=i, ticket=r["input"], label=r["label"],
                         baseline_b_pred=b["baseline_b_pred"], baseline_b_score=base,
                         ft_pred=p, ft_score=ft, delta=ft-base,
                         outcome="win" if ft > base else "loss" if ft < base else "tie"))
    rows.sort(key=lambda r: (r["delta"], r["ft_score"], r["i"]))
    (results / "qualitative.json").write_text(
        json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    losses = sum(r["outcome"] == "loss" for r in rows)
    print(f"Saved {len(rows)} full paired predictions; FT losses: {losses}.")
    if losses < 2:
        print("Report the actual loss count; do not invent examples to satisfy the rubric.")
"""
stages = STAGES.split()
assert stages and all(s in {"nb1", "nb2", "nb3", "nb4", "nb5", "nb6"} for s in stages)
for stage in stages:
    print(f"\nRunning {stage} with full evaluation", flush=True)
    subprocess.run([sys.executable, "-u", "-c", runner, stage], check=True)


Tesla T4 (cuda, sm_75, 14.6 GB) -> precision=fp16
  note: this GPU predates Ampere, so it has NO bfloat16. Using fp16 with gradient scaling instead. Tutorials that hardcode bf16=True fail here.

Running nb1 with full evaluation

Running nb2 with full evaluation

Running nb3 with full evaluation

Running nb4 with full evaluation

Running nb5 with full evaluation


In [4]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[  ok  ] unit tests                                       119 passed in 2.66s
[  ok  ] results/template_check.json                      
[  ok  ] results/mask_proof.json                          
[  ok  ] results/token_stats.json                         
[  ok  ] results/baselines_frozen.json                    
[  ok  ] results/runs.csv                                 
[  ok  ] results/verdict.json                             
[  ok  ] results/autopsy.json                             
[  ok  ] submission/REPORT.md                             
[ FAIL ] REPORT.md filled in                     

In [ ]:
# Kết nối với Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục và sao chép dữ liệu từ Colab vào Google Drive
import shutil
import os

source_dir = '/content/Day21-Track3-Finetuning-Lab'
dest_dir = '/content/drive/MyDrive/Day21-Track3-Finetuning-Lab'

if os.path.exists(source_dir):
    if not os.path.exists(dest_dir):
        shutil.copytree(source_dir, dest_dir)
        print(f"Đã sao chép thành công thư mục vào Google Drive tại: {dest_dir}")
    else:
        print(f"Thư mục đích đã tồn tại tại: {dest_dir}. Vui lòng kiểm tra hoặc xóa trước khi sao chép lại.")
else:
    print(f"Không tìm thấy thư mục nguồn: {source_dir}")

Mounted at /content/drive


## Ghi chú lần chạy cuối — 07/10/2026

Ngô Đinh Minh Nhật — 2A202602569.

Ô 3 đã được chạy lại trọn NB1 → NB5 với `EVAL_LIMIT=0` (đủ 50 target / 15 regression) và
max_length=256 (sửa tier T4 trong `src/labkit/config.py` bằng `sed` trước khi chạy; ô `sed` không
được lưu trong notebook). Mỗi stage chạy trong subprocess nên log chi tiết của NB1–NB5 không hiện ở đây;
kết quả nằm ở `results/`, gồm `baseline_predictions.json` (NB2, trước train) và `qualitative.json`
(50 cặp dự đoán (b) vs LoRA). Verdict vẫn FAILED vì regression.
